1. Setup

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)

from langchain.schema import Document

from ingestion.chunking import DocumentChunker
from ingestion.pdf_loader import PDFLoader
from utils.helpers import calculate_token_count

plt.style.use('seaborn-v0_8-darkgrid')

%matplotlib inline

2. Load Sample Document

In [ ]:
# Sample document for experimentation

sample_text = """
Artificial Intelligence and Machine Learning

Introduction to AI
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,
especially computer systems. These processes include learning, reasoning, and self-correction.

Machine Learning Basics
Machine learning is a subset of AI that provides systems the ability to automatically learn
and improve from experience without being explicitly programmed. Machine learning focuses
on the development of computer programs that can access data and use it to learn for themselves.

Deep Learning
Deep learning is part of a broader family of machine learning methods based on artificial
neural networks with representation learning. Learning can be supervised, semi-supervised,
or unsupervised.

Natural Language Processing
Natural language processing (NLP) is a subfield of linguistics, computer science, and
artificial intelligence concerned with the interactions between computers and human language.

Computer Vision
Computer vision is an interdisciplinary scientific field that deals with how computers
can gain high-level understanding from digital images or videos.

Robotics and AI
Robotics is a branch of engineering that involves the conception, design, manufacture,
and operation of robots.
"""

sample_doc = Document(page_content=sample_text)

print(f"Sample text length: {len(sample_text)} characters")
print(f"Token count: {calculate_token_count(sample_text)}")

3. Experiment 1 — Different Chunk Sizes

Generate Chunks with Different Sizes

In [ ]:
chunk_sizes = [100, 200, 500, 1000, 2000]
overlap = 50

results = []

for chunk_size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len
    )

    chunks = splitter.split_documents([sample_doc])

    avg_chunk_length = (
        sum(len(c.page_content) for c in chunks)
        / len(chunks)
    )

    avg_tokens = (
        sum(
            calculate_token_count(c.page_content)
            for c in chunks
        )
        / len(chunks)
    )

    results.append({
        "chunk_size": chunk_size,
        "num_chunks": len(chunks),
        "avg_length": avg_chunk_length,
        "avg_tokens": avg_tokens,
        "min_length": min(
            len(c.page_content)
            for c in chunks
        ),
        "max_length": max(
            len(c.page_content)
            for c in chunks
        )
    })

df_results = pd.DataFrame(results)

display(df_results)

Visualize Chunk Size Impact

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10)
)

# Number of Chunks
axes[0, 0].bar(
    df_results["chunk_size"],
    df_results["num_chunks"]
)

axes[0, 0].set_title(
    "Number of Chunks vs Chunk Size"
)

# Average Length
axes[0, 1].plot(
    df_results["chunk_size"],
    df_results["avg_length"],
    marker="o"
)

axes[0, 1].set_title(
    "Average Chunk Length"
)

# Average Tokens
axes[1, 0].plot(
    df_results["chunk_size"],
    df_results["avg_tokens"],
    marker="s"
)

axes[1, 0].set_title(
    "Average Token Count"
)

# Min-Max Range
x = range(len(df_results))

axes[1, 1].fill_between(
    x,
    df_results["min_length"],
    df_results["max_length"],
    alpha=0.3
)

axes[1, 1].plot(
    x,
    df_results["avg_length"],
    marker="o"
)

axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(
    df_results["chunk_size"]
)

axes[1, 1].set_title(
    "Chunk Length Range"
)

plt.tight_layout()
plt.show()

4. Experiment 2 — Different Overlap Sizes

Generate Results

In [ ]:
chunk_size = 500

overlaps = [0, 50, 100, 150, 200]

overlap_results = []

for overlap in overlaps:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len
    )

    chunks = splitter.split_documents([sample_doc])

    overlap_ratio = (
        overlap / chunk_size
    ) * 100

    overlap_results.append({
        "overlap": overlap,
        "overlap_ratio": overlap_ratio,
        "num_chunks": len(chunks),
        "avg_length": (
            sum(
                len(c.page_content)
                for c in chunks
            ) / len(chunks)
        )
    })

df_overlap = pd.DataFrame(
    overlap_results
)

display(df_overlap)

Visualize Overlap Impact

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

ax1.bar(
    df_overlap["overlap"],
    df_overlap["num_chunks"]
)

ax1.set_title(
    "Impact of Overlap on Chunk Count"
)

ax2.plot(
    df_overlap["overlap"],
    df_overlap["overlap_ratio"],
    marker="o"
)

ax2.set_title(
    "Overlap Ratio"
)

plt.tight_layout()
plt.show()

5. Experiment 3 — Splitting Strategy Comparison

In [ ]:
chunk_size = 500
overlap = 50

strategies = {

    "Recursive":
    RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=[
            "\n\n",
            "\n",
            " ",
            ""
        ]
    ),

    "Character":
    CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separator="\n"
    )
}

strategy_results = []

for name, splitter in strategies.items():

    chunks = splitter.split_documents(
        [sample_doc]
    )

    strategy_results.append({
        "strategy": name,
        "num_chunks": len(chunks),
        "avg_length":
            sum(
                len(c.page_content)
                for c in chunks
            ) / len(chunks),
        "std_length":
            pd.Series(
                [
                    len(c.page_content)
                    for c in chunks
                ]
            ).std(),
        "min_length":
            min(
                len(c.page_content)
                for c in chunks
            ),
        "max_length":
            max(
                len(c.page_content)
                for c in chunks
            )
    })

df_strategies = pd.DataFrame(
    strategy_results
)

display(df_strategies)

6. Experiment 4 — Chunk Content Analysis

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(
    [sample_doc]
)

print(
    f"Total chunks created: "
    f"{len(chunks)}"
)

for i, chunk in enumerate(
    chunks,
    start=1
):

    print("=" * 80)
    print(f"CHUNK {i}")
    print("=" * 80)

    print(
        f"Length: "
        f"{len(chunk.page_content)}"
    )

    print(
        f"Tokens: "
        f"{calculate_token_count(chunk.page_content)}"
    )

    print(
        chunk.page_content[:200]
    )

7. Experiment 5 — Token-Based Chunking

In [ ]:
from utils.helpers import (
    chunk_text_by_tokens
)

# Character-based chunking

char_splitter = (
    RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
)

char_chunks = (
    char_splitter.split_text(
        sample_text
    )
)

# Token-based chunking

token_chunks = (
    chunk_text_by_tokens(
        sample_text,
        max_tokens=100,
        overlap_tokens=10
    )
)

print("Character-based Chunking")
print(
    f"Chunks: "
    f"{len(char_chunks)}"
)

print(
    f"Avg Tokens: "
    f"{sum(calculate_token_count(c) for c in char_chunks) / len(char_chunks):.0f}"
)

print("\nToken-based Chunking")

print(
    f"Chunks: "
    f"{len(token_chunks)}"
)

print(
    f"Avg Tokens: "
    f"{sum(calculate_token_count(c) for c in token_chunks) / len(token_chunks):.0f}"
)

8. Recommendations

In [ ]:
print("=" * 80)
print(
    "CHUNKING RECOMMENDATIONS"
    .center(80)
)
print("=" * 80)

print("""
1. Chunk Size
   • 100–300 : High precision, lower context
   • 500–1000 : Recommended balance
   • 1500–2000 : More context, less precision

2. Overlap
   • 0 : Fastest processing
   • 50–100 : Low redundancy
   • 150–200 : Recommended
   • >250 : Excessive redundancy

3. Strategy
   • RecursiveCharacterTextSplitter : Recommended
   • CharacterTextSplitter : Simpler
   • Token-Based : Best for LLM limits

4. Optimal Configuration
   • Chunk Size = 1000 chars
   • Overlap = 200 chars
   • RecursiveCharacterTextSplitter
   • Separators:
     ["\\n\\n", "\\n", " ", ""]

5. Special Cases
   • Technical Docs → Larger chunks
   • Q&A Datasets → Smaller chunks
   • Legal Documents → Medium-large chunks
""")

9. Save Results

In [ ]:
from datetime import datetime

export_dir = (
    Path.cwd() / "exports"
)

export_dir.mkdir(
    exist_ok=True
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

df_results.to_csv(
    export_dir /
    f"chunk_size_experiment_{timestamp}.csv",
    index=False
)

df_overlap.to_csv(
    export_dir /
    f"overlap_experiment_{timestamp}.csv",
    index=False
)

df_strategies.to_csv(
    export_dir /
    f"strategy_comparison_{timestamp}.csv",
    index=False
)

print(
    f"✓ Results exported to "
    f"{export_dir}"
)